## 1) Importing libraries

In [ ]:
import tensorflow as tf
import numpy as np
import onnxruntime as ort

from PIL import Image

## 2) Getting model

In [57]:
MODEL_PATH = "./models/model.onnx"

session = ort.InferenceSession(MODEL_PATH)

# Obtém os nomes dos nós de entrada e saída do grafo ONNX
input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

## 3) Testing model

### 3.1) Using tensorflow

In [62]:
img = tf.keras.utils.load_img(
    path = "./chest_xray/pred/NORMAL/NORMAL_1.jpeg",
    color_mode = "grayscale",
    target_size = (200, 200),
    keep_aspect_ratio = True,
)

img = tf.keras.utils.img_to_array(img)

img = np.array(img, dtype=np.float32)
img = img / 0.255 
input_data = np.expand_dims(img, axis = 0)

raw_result = session.run([output_name], {input_name: input_data})

print(raw_result[0][0])

[9.9998271e-01 1.7252982e-05]


### 3.2) Using PIL

In [ ]:
img = Image.open(
    fp = "./chest_xray/pred/NORMAL/NORMAL_1.jpeg"
)

img = img.resize((200, 200))  # Redimensiona para o shape do modelo
        
# Converte para array numpy e normaliza (1/.255 ou /255 conforme seu Rescaling)
img_array = np.array(img, dtype=np.float32)
img_array = img_array / 0.255  # Exatamente o seu tf.keras.layers.Rescaling(1/.255)
        
# Adiciona as dimensões de Batch e Canal: de (200, 200) para (1, 200, 200, 1)
input_data = np.expand_dims(img_array, axis=0)
input_data = np.expand_dims(input_data, axis=-1)
        
# 5. Executa a inferência no ONNX Runtime
raw_result = session.run([output_name], {input_name: input_data})
probabilities = raw_result[0][0]

print(raw_result[0][0])

[0.9974521  0.00254796]
